### Profile Scrape

In [ ]:
def get_posts(target_username, login_username=None, login_password=None, top_k=1, download_dir="downloads"):
    """
    Extract top_k Instagram posts.
    Saves images locally and returns file paths.

    Returns: list of dicts with caption, metadata, images (file paths)
    """

    import os
    import instaloader
    import requests
    from itertools import islice
    from pathlib import Path

    # Create base download directory
    base_path = Path(download_dir) / target_username
    base_path.mkdir(parents=True, exist_ok=True)

    L = instaloader.Instaloader()

    if login_username and login_password:
        L.login(login_username, login_password)

    profile = instaloader.Profile.from_username(L.context, target_username)

    posts_data = []

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    for idx, post in enumerate(islice(profile.get_posts(), top_k), start=1):

        caption = post.caption
        metadata = post._node  # internal structure (works but unofficial)

        image_urls = []

        if post.typename == "GraphImage":
            image_urls.append(post.url)

        elif post.typename == "GraphSidecar":
            for node in post.get_sidecar_nodes():
                image_urls.append(node.display_url)

        # Folder per post
        post_folder = base_path / f"post_{idx}_{post.shortcode}"
        post_folder.mkdir(parents=True, exist_ok=True)

        image_paths = []

        for img_idx, url in enumerate(image_urls, start=1):
            try:
                resp = session.get(url, timeout=10)
                resp.raise_for_status()

                file_path = post_folder / f"image_{img_idx}.jpg"

                with open(file_path, "wb") as f:
                    f.write(resp.content)

                image_paths.append(file_path.as_posix())

            except Exception as e:
                print(f"Failed to download {url}: {e}")

        posts_data.append({
            "caption": caption,
            "metadata": metadata,
            "images": image_paths
        })

    return posts_data


In [39]:
posts_data = get_posts(target_username="maxverstappen1", top_k=10)

In [45]:
posts_data[9]["caption"]

'Leaving Abu Dhabi with our heads held high 🏆\n\n#F1 #RedBullRacing #AbuDhabiGP'

In [48]:
posts_data[9]["images"]

[]

In [49]:
posts_data[9]["metadata"]

{'__typename': 'GraphVideo',
 'id': '3782539475751324614',
 'shortcode': 'DR-R5zJDMvG',
 'dimensions': {'height': 1333, 'width': 750},
 'display_url': 'https://instagram.fbom19-5.fna.fbcdn.net/v/t51.2885-15/576557832_18540090484033142_1704750260336906773_n.jpg?stp=dst-jpg_e15_tt6&_nc_ht=instagram.fbom19-5.fna.fbcdn.net&_nc_cat=102&_nc_oc=Q6cZ2QEVx0bTiBRhOMWqo9dJbxglHsCfK3cjMLtmWOSV1S_O888vViGgI90KOC4i8fM4qBdbaKOh25HLLaiZQAhyIXz0&_nc_ohc=9lffnrEjcEwQ7kNvwFuh9DC&_nc_gid=p3oSBNDLoyKDWeiMySzxNg&edm=AOQ1c0wBAAAA&ccb=7-5&oh=00_Afpy_Erg7-fdTEOF3H5TAbtnRhhmJNYQtBwTwTVhIiAq-g&oe=697D1B01&_nc_sid=8b3546',
 'edge_media_to_tagged_user': {'edges': [{'node': {'user': {'full_name': 'Max Verstappen',
      'followed_by_viewer': False,
      'id': '43904777',
      'is_verified': True,
      'profile_pic_url': 'https://instagram.fbom19-2.fna.fbcdn.net/v/t51.2885-19/428583865_720195150284958_3586880092673320798_n.jpg?stp=dst-jpg_e0_s150x150_tt6&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmM

### Geolocation Prediction

### Face Detection

In [1]:
import cv2
import os
from insightface.utils import face_align
from insightface.app import FaceAnalysis

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
face_analyzer = FaceAnalysis(name='antelopev2', providers=providers)
face_analyzer.prepare(ctx_id=0, det_size=(640, 640))

def detect_faces_from_image_path(image_path, face_analyzer):
    """
    Detect faces from an image path using InsightFace.

    Returns a list of dicts with:
    - face_img      : aligned face (BGR)
    - display_img  : raw cropped face (BGR)
    - bbox         : (x, y, w, h)
    - embedding    : ArcFace embedding (np.float32)
    - det_score    : detection confidence
    """

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    frame = cv2.imread(image_path)
    if frame is None:
        raise ValueError(f"Failed to load image: {image_path}")

    h, w = frame.shape[:2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = []

    try:
        faces = face_analyzer.get(rgb_frame)

        for face in faces:
            x1, y1, x2, y2 = face.bbox.astype(int)

            # Clip bbox
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            display_img = frame[y1:y2, x1:x2]

            # Alignment
            if face.kps is not None:
                aligned = face_align.norm_crop(rgb_frame, face.kps)
                face_img = cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR)
            else:
                face_img = display_img

            results.append({
                "face_img": face_img,
                "display_img": display_img,
                "bbox": (x1, y1, x2 - x1, y2 - y1),
                "embedding": face.embedding.astype("float32"),
                "det_score": float(face.det_score)
            })

        return results

    except Exception as e:
        print(f"[Face Detection Error] {e}")
        return []


d:\Projects\Research Work\Safe Scroll\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\bryan/.insightface\models\antelopev2\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider

In [2]:
faces = detect_faces_from_image_path("1.jpg", face_analyzer)

for i, f in enumerate(faces):
    print(f"Face {i}: score={f['det_score']}, bbox={f['bbox']}")


d:\Projects\Research Work\Safe Scroll\venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Face 0: score=0.8822408318519592, bbox=(np.int64(35), np.int64(125), np.int64(33), np.int64(40))
Face 1: score=0.8744627237319946, bbox=(np.int64(181), np.int64(154), np.int64(29), np.int64(34))
Face 2: score=0.8635662198066711, bbox=(np.int64(533), np.int64(79), np.int64(21), np.int64(23))
Face 3: score=0.8550218343734741, bbox=(np.int64(216), np.int64(101), np.int64(18), np.int64(25))
Face 4: score=0.8458254337310791, bbox=(np.int64(305), np.int64(93), np.int64(20), np.int64(22))
Face 5: score=0.8412182331085205, bbox=(np.int64(68), np.int64(74), np.int64(17), np.int64(19))
Face 6: score=0.8214007616043091, bbox=(np.int64(188), np.int64(118), np.int64(27), np.int64(30))
Face 7: score=0.8196161985397339, bbox=(np.int64(204), np.int64(63), np.int64(18), np.int64(20))
Face 8: score=0.8107147216796875, bbox=(np.int64(99), np.int64(109), np.int64(27), np.int64(35))
Face 9: score=0.8084000945091248, bbox=(np.int64(81), np.int64(169), np.int64(30), np.int64(40))
Face 10: score=0.79651218652

In [3]:
import cv2
import os

def save_faces_with_bboxes(
    image_path,
    face_analyzer,
    output_path="output.jpg"
):
    """
    Detect faces in an image and save the image with bounding boxes drawn.

    Args:
        image_path (str): input image path
        face_analyzer: initialized InsightFace FaceAnalysis
        output_path (str): optional output path

    Returns:
        str: path to saved image
    """

    faces = detect_faces_from_image_path(image_path, face_analyzer)

    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Failed to load image")

    for face in faces:
        x, y, w, h = face["bbox"]
        score = face["det_score"]

        cv2.rectangle(
            image,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            image,
            f"{score:.2f}",
            (x, y - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1
        )

    if output_path is None:
        base, ext = os.path.splitext(image_path)
        output_path = f"{base}_faces{ext}"

    cv2.imwrite(output_path, image)
    print(f"Saved to {output_path}")


In [4]:
save_faces_with_bboxes("1.jpg", face_analyzer)

Saved to output.jpg


### VLLM + LLM

In [4]:
from ollama import chat

response = chat(
    model='llama3.1:8b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! How are you today? Is there something I can help you with or would you like to chat?


In [3]:
from ollama import chat

response = chat(
    model='granite3.3:8b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hello! How can I assist you today?


In [18]:
from ollama import chat

response = chat(
    model='gemma3:4b',
    messages=[
        {
            'role': 'user',
            'content': 'Does it look like morning, afternoon or night in the image?',
            'images': ['face_images/input.jpg']
        }
    ]
)

print(response.message.content)


Based on the lighting in the image, it looks like **afternoon**. The light is bright and relatively even, suggesting it's not early morning or late afternoon/evening when the light is softer and more golden.


In [21]:
def context_aware_ocr(image_path: list[str], model: str = 'gemma3:4b'):
    try:
        response = chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": "Act as a context-aware OCR model and give me all the text you see in this image",
                    "images": [image_path]
                }
            ]
        )

        return response["message"]["content"]
    except Exception as e:
        print(f"Error occured: {e}")
        return "Error occured while performing OCR, proceed without it."

context_aware_ocr("images/documents.jpg")

'Okay, here’s the text I\'ve extracted from the document in the image.  Please note that OCR accuracy can be affected by image quality, font, and skew.\n\n---\n\n**Last Year Earnings**\n\nThe last year earnings shows a significant increase in revenue due to a higher level of sales and the successful implementation of a new marketing strategy. In addition, cost-cutting measures and operational efficiencies contributed to improved profitability.\n\nOur overall revenue increased by 15% compared to the previous year, driven primarily by strong demand for our flagship product, "Alpha". The growth in sales was fueled by expanded distribution channels, including online sales and strategic partnerships with key retailers. We invested in product development and innovation to enhance our product offerings and maintain a competitive advantage.\n\nCost control efforts resulted in a 10% reduction in operating expenses, mainly through streamlining processes and negotiating favorable supplier contrac